## 7c. AI Agent as a Predictive Model

AI agent used for held-out predictions: ChatGPT (GPT-5.5 Thinking). The full held-out prompt is saved as `ai_heldout_prompt.txt`, and the returned predictions are saved as `ai_heldout_predictions.csv`.

This section evaluates the AI agent on held-out real search queries and compares its predictive performance to the MNL model from Problem 1.

In [19]:
import pandas as pd
import numpy as np

real_df = pd.read_csv("data (1).csv")

choice_col_real = "booking_bool"

real_df["alt_id"] = real_df.groupby("srch_id").cumcount() + 1

continuous_features = [
    "prop_starrating",
    "prop_review_score",
    "prop_location_score",
    "prop_accesibility_score",
    "prop_log_historical_price",
    "price_usd"
]

binary_features = [
    "prop_brand_bool",
    "promotion_flag"
]

features = continuous_features + binary_features

In [20]:
rng = np.random.default_rng(42)

all_srch_ids = real_df["srch_id"].unique()

context_srch_ids = rng.choice(all_srch_ids, size=10, replace=False)

remaining_srch_ids = np.setdiff1d(all_srch_ids, context_srch_ids)

heldout_srch_ids = rng.choice(remaining_srch_ids, size=50, replace=False)

context_df = real_df[real_df["srch_id"].isin(context_srch_ids)].copy()
heldout_df = real_df[real_df["srch_id"].isin(heldout_srch_ids)].copy()

print("Context queries:", context_df["srch_id"].nunique())
print("Held-out queries:", heldout_df["srch_id"].nunique())
print("Held-out rows:", len(heldout_df))

Context queries: 10
Held-out queries: 50
Held-out rows: 868


In [21]:
def get_true_choice(group, choice_col):
    booked = group[group[choice_col] == 1]

    if len(booked) == 1:
        return int(booked.iloc[0]["alt_id"])
    elif len(booked) == 0:
        return "NO_PURCHASE"
    else:
        raise ValueError(f"Search {group['srch_id'].iloc[0]} has multiple bookings.")

true_choices = (
    heldout_df
    .groupby("srch_id")
    .apply(lambda g: get_true_choice(g, choice_col_real))
    .reset_index(name="true_choice")
)

true_choices.head()

/tmp/ipykernel_1999/165081072.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: get_true_choice(g, choice_col_real))


,srch_id,true_choice
0,43682,11
1,61359,1
2,65339,6
3,74418,3
4,88597,7


In [22]:
def summarize_query_for_prompt(group, include_answer=True):
    srch_id = group["srch_id"].iloc[0]

    context_cols = [
        "srch_booking_window",
        "srch_adults_count",
        "srch_children_count",
        "srch_room_count",
        "srch_saturday_night_bool",
    ]

    available_context_cols = [c for c in context_cols if c in group.columns]

    hotel_cols = ["alt_id"] + features
    available_hotel_cols = [c for c in hotel_cols if c in group.columns]

    context_text = group.iloc[0][available_context_cols].to_dict()
    hotel_table = group[available_hotel_cols].to_string(index=False)

    prompt_part = f"""
Search query {srch_id}
Customer/search context:
{context_text}

Hotel options:
{hotel_table}
""".strip()

    if include_answer:
        true_choice = get_true_choice(group, choice_col_real)
        prompt_part += f"\nObserved customer choice: {true_choice}"

    return prompt_part

In [23]:
context_examples = []

for srch_id, group in context_df.groupby("srch_id"):
    context_examples.append(summarize_query_for_prompt(group, include_answer=True))

heldout_prompts = []

for srch_id, group in heldout_df.groupby("srch_id"):
    heldout_prompts.append(summarize_query_for_prompt(group, include_answer=False))

full_prompt = f"""
You are predicting hotel booking choices.

Below are examples from real hotel search queries. Each example includes the customer/search context, the hotel alternatives shown, and the actual observed customer choice. The observed choice is either one hotel alt_id or NO_PURCHASE.

Context examples:
{"\n\n".join(context_examples)}

Now predict the customer choice for each held-out search query below. For each query, return exactly one prediction:
- one listed alt_id, or
- NO_PURCHASE

Return your answer as a CSV with columns:
srch_id,predicted_choice

Held-out queries:
{"\n\n".join(heldout_prompts)}
""".strip()

# print(full_prompt)
with open("ai_heldout_prompt.txt", "w", encoding="utf-8") as f:
    f.write(full_prompt)

## Comparison

In [24]:
ai_predictions = pd.read_csv("ai_heldout_predictions.csv")

ai_predictions["predicted_choice"] = ai_predictions["predicted_choice"].astype(str)
true_choices["true_choice"] = true_choices["true_choice"].astype(str)

ai_eval = true_choices.merge(ai_predictions, on="srch_id")

ai_eval["ai_correct"] = (
    ai_eval["true_choice"] == ai_eval["predicted_choice"]
)

ai_accuracy = ai_eval["ai_correct"].mean()

print("AI held-out query accuracy:", ai_accuracy)
# ai_eval

AI held-out query accuracy: 0.18


I use query-level exact-choice accuracy as the first metric. A prediction is counted as correct if it matches the exact booked hotel alternative, or correctly predicts `NO_PURCHASE` when the customer did not book.

In [25]:
# Human MNL normalized coefficients from Problem 1
human_beta = {
    "intercept": -1.9815321907864278,
    "prop_starrating": 0.4081249536151655,
    "prop_review_score": 0.10876096623704055,
    "prop_brand_bool": 0.22992269948768013,
    "prop_location_score": 0.02202632301274303,
    "prop_accesibility_score": 0.04344412341249515,
    "prop_log_historical_price": -0.06686945209512846,
    "price_usd": -1.3311099651462353,
    "promotion_flag": 0.45402977040295234
}

# Use non-held-out data to compute scaling for continuous features only
train_df = real_df[~real_df["srch_id"].isin(heldout_srch_ids)].copy()

continuous_means = train_df[continuous_features].mean()
continuous_stds = train_df[continuous_features].std().replace(0, 1)

continuous_medians = train_df[continuous_features].median()
binary_medians = train_df[binary_features].median()

def prepare_mnl_features(group):
    g = group.copy()

    # Clean and standardize continuous features
    g[continuous_features] = (
        g[continuous_features]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(continuous_medians)
    )

    # Clean binary features, but do not standardize them
    g[binary_features] = (
        g[binary_features]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(binary_medians)
    )

    X = pd.DataFrame(index=g.index)

    X[continuous_features] = (
        g[continuous_features] - continuous_means
    ) / continuous_stds

    X[binary_features] = g[binary_features]

    return g, X

def compute_mnl_utilities(group, beta):
    g, X = prepare_mnl_features(group)

    u = pd.Series(beta["intercept"], index=g.index, dtype=float)

    for f in continuous_features:
        u += beta[f] * X[f]

    for f in binary_features:
        u += beta[f] * X[f]

    return g, u

def mnl_predict_query_fixed(group, beta):
    g, u = compute_mnl_utilities(group, beta)

    max_idx = u.idxmax()
    max_u = u.loc[max_idx]

    # Outside option utility is normalized to 0
    if max_u <= 0:
        return "NO_PURCHASE"
    else:
        return str(int(g.loc[max_idx, "alt_id"]))

mnl_predictions = []

for srch_id, group in heldout_df.groupby("srch_id"):
    pred = mnl_predict_query_fixed(group, human_beta)
    mnl_predictions.append({
        "srch_id": srch_id,
        "mnl_predicted_choice": pred
    })

mnl_predictions = pd.DataFrame(mnl_predictions)

# Add always-NO_PURCHASE baseline
baseline_predictions = true_choices[["srch_id"]].copy()
baseline_predictions["baseline_predicted_choice"] = "NO_PURCHASE"

# Merge AI, MNL, baseline, and true outcomes
eval_df = (
    true_choices
    .merge(ai_predictions, on="srch_id")
    .merge(mnl_predictions, on="srch_id")
    .merge(baseline_predictions, on="srch_id")
)

eval_df["true_choice"] = eval_df["true_choice"].astype(str)
eval_df["predicted_choice"] = eval_df["predicted_choice"].astype(str)
eval_df["mnl_predicted_choice"] = eval_df["mnl_predicted_choice"].astype(str)
eval_df["baseline_predicted_choice"] = eval_df["baseline_predicted_choice"].astype(str)

eval_df["ai_correct"] = eval_df["true_choice"] == eval_df["predicted_choice"]
eval_df["mnl_correct"] = eval_df["true_choice"] == eval_df["mnl_predicted_choice"]
eval_df["baseline_correct"] = eval_df["true_choice"] == eval_df["baseline_predicted_choice"]

accuracy_table = pd.DataFrame({
    "Model": ["AI agent", "MNL", "Always NO_PURCHASE baseline"],
    "Held-out query accuracy": [
        eval_df["ai_correct"].mean(),
        eval_df["mnl_correct"].mean(),
        eval_df["baseline_correct"].mean()
    ],
    "Correct predictions": [
        eval_df["ai_correct"].sum(),
        eval_df["mnl_correct"].sum(),
        eval_df["baseline_correct"].sum()
    ],
    "Held-out queries": [
        len(eval_df),
        len(eval_df),
        len(eval_df)
    ]
})

accuracy_table

,Model,Held-out query accuracy,Correct predictions,Held-out queries
0,AI agent,0.18,9,50
1,MNL,0.26,13,50
2,Always NO_PURCHASE baseline,0.26,13,50


In [26]:
from statsmodels.stats.proportion import proportion_confint

ci_rows = []

for model_name, correct_col in [
    ("AI agent", "ai_correct"),
    ("MNL", "mnl_correct"),
    ("Always NO_PURCHASE baseline", "baseline_correct")
]:
    correct = int(eval_df[correct_col].sum())
    n = len(eval_df)
    acc = correct / n

    ci_low, ci_high = proportion_confint(
        count=correct,
        nobs=n,
        alpha=0.05,
        method="wilson"
    )

    ci_rows.append({
        "Model": model_name,
        "Accuracy": acc,
        "Correct": correct,
        "N": n,
        "95% CI lower": ci_low,
        "95% CI upper": ci_high
    })

accuracy_ci_table = pd.DataFrame(ci_rows)
accuracy_ci_table

,Model,Accuracy,Correct,N,95% CI lower,95% CI upper
0,AI agent,0.18,9,50,0.097702,0.307961
1,MNL,0.26,13,50,0.158715,0.395532
2,Always NO_PURCHASE baseline,0.26,13,50,0.158715,0.395532


Exact-choice accuracy is a strict metric because each search query contains many hotel alternatives. A model only receives credit if it predicts the exact booked hotel or correctly predicts no-purchase. I also include an always-`NO_PURCHASE` baseline because a meaningful share of held-out queries result in no booking. This baseline helps show whether the AI agent or MNL model adds predictive value beyond simply predicting the outside option.

In [30]:
import pandas as pd
import numpy as np

features_to_compare = [
    "price_usd",
    "prop_starrating",
    "prop_review_score",
    "prop_location_score",
    "prop_brand_bool",
    "promotion_flag"
]

# Make sure choice columns are strings
true_choices["true_choice"] = true_choices["true_choice"].astype(str)
ai_predictions["predicted_choice"] = ai_predictions["predicted_choice"].astype(str)
mnl_predictions["mnl_predicted_choice"] = mnl_predictions["mnl_predicted_choice"].astype(str)
baseline_predictions["baseline_predicted_choice"] = baseline_predictions["baseline_predicted_choice"].astype(str)

def get_chosen_rows(pred_df, pred_col, label):
    rows = []

    for _, row in pred_df.iterrows():
        srch_id = row["srch_id"]
        choice = str(row[pred_col])

        if choice == "NO_PURCHASE":
            rows.append({
                "srch_id": srch_id,
                "model": label,
                "choice": "NO_PURCHASE",
                "purchased": 0,
                **{f: np.nan for f in features_to_compare}
            })
        else:
            alt_id = int(float(choice))

            chosen = heldout_df[
                (heldout_df["srch_id"] == srch_id) &
                (heldout_df["alt_id"] == alt_id)
            ]

            if len(chosen) != 1:
                raise ValueError(
                    f"Could not find exactly one row for srch_id={srch_id}, alt_id={alt_id}. "
                    f"Found {len(chosen)} rows."
                )

            chosen = chosen.iloc[0]

            rows.append({
                "srch_id": srch_id,
                "model": label,
                "choice": str(alt_id),
                "purchased": 1,
                **{f: chosen[f] for f in features_to_compare}
            })

    return pd.DataFrame(rows)

observed_behavior = get_chosen_rows(
    true_choices.rename(columns={"true_choice": "choice"}),
    "choice",
    "Observed human data"
)

ai_behavior = get_chosen_rows(
    ai_predictions,
    "predicted_choice",
    "AI agent"
)

mnl_behavior = get_chosen_rows(
    mnl_predictions,
    "mnl_predicted_choice",
    "MNL hard-choice"
)

baseline_behavior = get_chosen_rows(
    baseline_predictions.rename(columns={"baseline_predicted_choice": "choice"}),
    "choice",
    "Always NO_PURCHASE baseline"
)

behavior_rows = pd.concat(
    [observed_behavior, ai_behavior, mnl_behavior, baseline_behavior],
    ignore_index=True
)

summary = (
    behavior_rows
    .groupby("model")
    .agg(
        purchase_rate=("purchased", "mean"),
        no_purchase_rate=("purchased", lambda x: 1 - x.mean()),
        num_predicted_purchases=("purchased", "sum"),
        avg_chosen_price=("price_usd", "mean"),
        avg_chosen_star_rating=("prop_starrating", "mean"),
        avg_chosen_review_score=("prop_review_score", "mean"),
        avg_chosen_location_score=("prop_location_score", "mean"),
        share_chosen_brand=("prop_brand_bool", "mean"),
        share_chosen_promotion=("promotion_flag", "mean")
    )
    .reset_index()
)

# Make a display version where undefined chosen-hotel metrics are shown as N/A
chosen_metric_cols = [
    "avg_chosen_price",
    "avg_chosen_star_rating",
    "avg_chosen_review_score",
    "avg_chosen_location_score",
    "share_chosen_brand",
    "share_chosen_promotion"
]

summary_display = summary.copy()

for col in chosen_metric_cols:
    summary_display[col] = np.where(
        summary_display["num_predicted_purchases"] == 0,
        "N/A",
        summary_display[col].round(3).astype(str)
    )

summary_display["purchase_rate"] = summary_display["purchase_rate"].round(3)
summary_display["no_purchase_rate"] = summary_display["no_purchase_rate"].round(3)
summary_display["num_predicted_purchases"] = summary_display["num_predicted_purchases"].astype(int)

summary_display

,model,purchase_rate,no_purchase_rate,num_predicted_purchases,avg_chosen_price,avg_chosen_star_rating,avg_chosen_review_score,avg_chosen_location_score,share_chosen_brand,share_chosen_promotion
0,AI agent,0.82,0.18,41,136.829,3.707,4.927,3.0,0.902,0.317
1,Always NO_PURCHASE baseline,0.00,1.00,0,N/A,N/A,N/A,N/A,N/A,N/A
2,MNL hard-choice,0.00,1.00,0,N/A,N/A,N/A,N/A,N/A,N/A
3,Observed human data,0.74,0.26,37,115.081,3.243,4.135,2.595,0.784,0.243


### Behavioral comparison using hard predictions

Exact-choice accuracy is a strict metric because each search query contains many hotel alternatives, and a model only receives credit if it predicts the exact booked hotel or correctly predicts no-purchase. To complement exact accuracy, I also compare the aggregate behavior implied by each model to the behavior observed in the held-out data.

The behavioral comparison shows whether each model produces choices with similar aggregate characteristics to actual human choices. In particular, I compare purchase rate, no-purchase rate, average chosen price, average chosen star rating, average chosen review score, average chosen location score, brand share, and promotion share.

The `N/A` values for the always-`NO_PURCHASE` baseline and the MNL hard-choice model occur because both predict no-purchase for all 50 held-out queries. Since neither selects any hotel alternatives under the hard-choice rule, there are no chosen-hotel attributes to average. These rows are still useful for comparing purchase/no-purchase behavior, but not for comparing chosen-hotel characteristics.

This table should be interpreted alongside the exact-choice accuracy table. A model can miss the exact realized hotel but still produce choices with similar observable characteristics to human choices. Conversely, a model may get some exact matches while still implying unrealistic aggregate behavior. Because the MNL hard-choice rule discards the rest of the probability distribution, I also report the MNL probability-implied behavior below.

In [31]:
def mnl_probabilities_for_query_fixed(group, beta):
    g, u = compute_mnl_utilities(group, beta)

    # Numerical stability with outside option utility = 0
    max_u = max(0, np.max(u))
    exp_u = np.exp(u - max_u)
    exp_outside = np.exp(0 - max_u)

    denom = exp_outside + exp_u.sum()

    hotel_probs = exp_u / denom
    no_purchase_prob = exp_outside / denom

    return hotel_probs, no_purchase_prob


# Aggregate MNL probability-implied behavior across all held-out queries
mnl_prob_rows = []

for srch_id, group in heldout_df.groupby("srch_id"):
    hotel_probs, no_purchase_prob = mnl_probabilities_for_query_fixed(
      group,
      human_beta
    )

    temp = group.copy()
    temp["mnl_prob"] = hotel_probs
    temp["no_purchase_prob"] = no_purchase_prob

    mnl_prob_rows.append(temp)

mnl_prob_df = pd.concat(mnl_prob_rows, ignore_index=True)

total_purchase_prob = mnl_prob_df["mnl_prob"].sum()
num_queries = heldout_df["srch_id"].nunique()

mnl_expected_summary = pd.DataFrame({
    "model": ["MNL probability-implied"],
    "purchase_rate": [total_purchase_prob / num_queries],
    "no_purchase_rate": [1 - total_purchase_prob / num_queries],
    "avg_chosen_price": [
        (mnl_prob_df["mnl_prob"] * mnl_prob_df["price_usd"]).sum() / total_purchase_prob
    ],
    "avg_chosen_star_rating": [
        (mnl_prob_df["mnl_prob"] * mnl_prob_df["prop_starrating"]).sum() / total_purchase_prob
    ],
    "avg_chosen_review_score": [
        (mnl_prob_df["mnl_prob"] * mnl_prob_df["prop_review_score"]).sum() / total_purchase_prob
    ],
    "avg_chosen_location_score": [
        (mnl_prob_df["mnl_prob"] * mnl_prob_df["prop_location_score"]).sum() / total_purchase_prob
    ],
    "share_chosen_brand": [
        (mnl_prob_df["mnl_prob"] * mnl_prob_df["prop_brand_bool"]).sum() / total_purchase_prob
    ],
    "share_chosen_promotion": [
        (mnl_prob_df["mnl_prob"] * mnl_prob_df["promotion_flag"]).sum() / total_purchase_prob
    ]
})

behavior_summary_with_expected_mnl = pd.concat(
    [summary, mnl_expected_summary],
    ignore_index=True
)

behavior_summary_with_expected_mnl

,model,purchase_rate,no_purchase_rate,num_predicted_purchases,avg_chosen_price,avg_chosen_star_rating,avg_chosen_review_score,avg_chosen_location_score,share_chosen_brand,share_chosen_promotion
0,AI agent,0.820000,0.180000,41.0,136.829268,3.707317,4.926829,3.000000,0.902439,0.317073
1,Always NO_PURCHASE baseline,0.000000,1.000000,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,MNL hard-choice,0.000000,1.000000,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,Observed human data,0.740000,0.260000,37.0,115.081081,3.243243,4.135135,2.594595,0.783784,0.243243
4,MNL probability-implied,0.684909,0.315091,NaN,124.550448,3.200549,3.988601,2.353204,0.764838,0.190832


### MNL probability-implied behavior

The MNL model naturally outputs probabilities for each hotel alternative and for the no-purchase option. The hard-choice prediction discards most of this information by selecting only the single highest-probability outcome. To use the full probability distribution, I also summarize the MNL-implied behavior in expectation.

For each query, the MNL-implied purchase probability is the sum of the probabilities assigned to all hotel alternatives. The expected chosen-hotel attributes are computed as probability-weighted averages of hotel attributes, conditional on purchase.

This probability-implied version gives a more reasonable picture of the MNL model. The hard-choice MNL predicts purchase in 0% of held-out queries, while the probability-implied MNL purchase rate is about 68.5%, much closer to the observed 74.0%. Its expected chosen-hotel price, star rating, review score, brand share, and promotion share are also much closer to observed behavior than the hard-choice MNL summary. This shows why relying only on the highest-probability choice can lose useful information in a discrete choice model.

### Interpretation

The exact-choice accuracy metric is strict: a prediction is only correct if it matches the exact booked hotel or correctly predicts no-purchase. In this held-out sample, the always-`NO_PURCHASE` baseline is important because 26% of held-out queries resulted in no purchase. The AI agent, MNL hard-choice rule, and baseline should therefore be compared cautiously, especially because the held-out set contains only 50 search queries and the confidence intervals are wide.

On exact-choice accuracy, the AI agent predicted 9 out of 50 held-out queries correctly, while both the MNL hard-choice rule and the always-`NO_PURCHASE` baseline predicted 13 out of 50 correctly. This means the MNL hard-choice rule did not outperform the trivial no-purchase baseline on this held-out sample, even though both scored higher than the AI agent on strict exact-choice accuracy. The behavioral metrics provide a more nuanced comparison. The AI agent better matches the observed purchase/no-purchase rate than the MNL hard-choice rule: observed customers purchased in 74% of queries, while the AI predicted purchases in 82% of queries and the MNL hard-choice rule predicted purchases in 0% of queries. However, the MNL probability-implied summary shows that the full MNL probability distribution is much closer to observed behavior than its hard prediction alone, with an implied purchase rate of about 68.5%.

Overall, the AI agent captures some broad behavioral patterns and produces plausible hard choices, but the MNL model remains useful because it provides a full probability distribution over choices rather than only a single prediction. These results should be interpreted as illustrative rather than definitive. A larger held-out set or repeated random splits would provide a more stable comparison.